# Build a Notes App Model

Combine Kotlin data, functions, classes, and collections into a small model that you can test before building an Android screen.

This final lesson introduces no new Kotlin syntax. You will use earlier skills to add, find, update, and remove notes, then build a related reminder model with less starter code. The model keeps its data only while the program is running.

## Learning Goals

- Combine earlier Kotlin features into a model that adds, finds, updates, and removes notes.
- Handle missing IDs safely and preserve records that an operation should not change.
- Test a complete model using normal cases, missing records, and a sequence of changes.

## Why This Matters

A notes screen needs reliable behavior beneath its buttons. Editing one title should not change another note. Looking for a deleted note should produce a useful missing result instead of a crash.

You can test these rules with ordinary Kotlin before adding an Android interface. That keeps the feedback clear: the printed results tell you which data rule worked. Later, a screen can call these same kinds of operations in response to user actions. This lesson does not build that screen or add permanent storage.

## Check Your Starting Point

Recall two earlier distinctions. Lesson 3's `copy` creates another data-class instance; Lesson 5's `apply` configures and returns its receiver. Which operation would help create a replacement note while keeping an existing note object unchanged? Then explain how Lesson 4's `find` reports that no record matched.

In [ ]:
Your response:
Write your explanation here.

<details>
<summary>Show answer</summary>

Use `copy` to create a replacement instance with selected property values. `apply` would configure the same receiver instead. `find` returns null when no element matches, so a nullable return type and explicit missing-value handling fit a model search.

</details>

## Video Demonstration

Build and run a complete notes model, then interpret the results of editing and deleting one note. Use the walkthrough to prepare for your independent reminder model.

<video style="max-width:100%;height:auto;" controls preload="metadata" width="800" aria-label="Build a Notes App Model demonstration">
<source src="media/06_build_a_notes_app_model/lesson.mp4" type="video/mp4">
<track kind="captions" src="media/06_build_a_notes_app_model/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript and visual description](media/06_build_a_notes_app_model/transcript.md).

## Concept

### Reuse the Pieces You Already Know

A model represents the data and rules behind an app feature. Our notes model needs only a few familiar pieces:

| Earlier skill | Job in this model |
| --- | --- |
| `data class` and constructor properties | Represent a note's integer ID and title |
| `private` and class methods | Keep the list inside the store and expose named operations |
| `mutableListOf<Note>()`, `add`, and `remove` | Hold and change the current group of notes |
| `find` and a lambda predicate | Locate a note whose ID matches the request |
| Nullable types, a null check, and `?.` / `?:` | Handle a request that has no matching record |
| `copy` with a named argument | Create a replacement note with a changed title |

We will first trace the collection operations directly. Then we will put them behind class methods. Each notebook defines its own classes; it does not use definitions left over from another lesson.

### State the Data Rules First

An ID identifies a note independently of its title. A title can change while the ID stays the same. For this exercise, **the caller must supply a unique ID for each added note**. The store does not generate IDs or reject duplicates. Testing only unique IDs follows that stated requirement; it does not prove duplicate checking exists.

- Add appends the supplied note.
- Find returns the first matching note, or null when the ID is missing.
- Update replaces the matching record with a copy containing the new title. A missing ID changes nothing.
- Remove deletes the matching record. A missing ID changes nothing.
- Each change must preserve notes with other IDs.

Data stays in memory. Starting a new script run or creating a new store begins with an empty list. The store does not write to the device, a database, or a server.

### Find a Record Before Changing It

The two records below have different IDs. The predicate compares each record's `id` with the requested number. A missing search returns null, which becomes `Missing` only at the printing step.

In [ ]:
data class ModelNote(val id: Int, val title: String)
val modelNotes = mutableListOf(ModelNote(7, "Groceries"), ModelNote(8, "Trip"))
println(modelNotes.find { it.id == 7 }?.title ?: "Missing")
println(modelNotes.find { it.id == 99 }?.title ?: "Missing")

This prints `Groceries` and `Missing`. Searching by ID lets us find the same record even after its title changes. The missing search does not add or remove anything.

### Replace the Matching Record

A note's `val title` cannot be reassigned. We can create a copy with the new title, remove the old record from the mutable list, and add the replacement. The list changes; the old note object itself does not.

The stable local `beforeEdit` lets the compiler narrow its nullable type inside the successful null check, just as in Lesson 2.

In [ ]:
val beforeEdit = modelNotes.find { it.id == 7 }
if (beforeEdit != null) {
    modelNotes.remove(beforeEdit)
    modelNotes.add(beforeEdit.copy(title = "Weekend groceries"))
}
println(beforeEdit?.title ?: "Missing")
println(modelNotes.find { it.id == 7 }?.title ?: "Missing")
println(modelNotes.find { it.id == 8 }?.title ?: "Missing")

The output is:

```text
Groceries
Weekend groceries
Trip
```

The retained reference still has the original title. Searching the list now finds the replacement for ID 7. ID 8 still has its original title.

This implementation removes and then appends. A replacement can therefore move to the end of the list. Our requirements do not promise a fixed display order; a later screen that needs one would need an explicit ordering rule. Do not infer that update preserves a record's list position.

### Make a Missing Change Harmless

When the search returns null, the condition is false and the change block is skipped. There is no need for `!!`. A missing record is an ordinary result that we designed for.

In [ ]:
val missingModelNote = modelNotes.find { it.id == 99 }
if (missingModelNote != null) {
    modelNotes.remove(missingModelNote)
}
println(modelNotes.size)
println(modelNotes.find { it.id == 7 }?.title ?: "Missing")
println(modelNotes.find { it.id == 8 }?.title ?: "Missing")

This prints `2`, `Weekend groceries`, and `Trip`. The attempted removal of ID 99 leaves both existing records unchanged. An update can use the same guard around its remove-and-add steps.

### Remove One Record and Check the Other

Use the same find-and-check pattern for an existing ID. After removal, a fresh search is necessary to learn whether the list still contains the record; an old variable can still refer to the removed object.

In [ ]:
val noteToRemove = modelNotes.find { it.id == 7 }
if (noteToRemove != null) {
    modelNotes.remove(noteToRemove)
}
println(modelNotes.find { it.id == 7 }?.title ?: "Missing")
println(modelNotes.find { it.id == 8 }?.title ?: "Missing")

The output is `Missing` and `Trip`. ID 7 is no longer in the list, while ID 8 remains. Removing a reference from the list is different from changing the fields of the removed object.

These short traces reuse the selection, copying, and null-handling mechanisms from earlier lessons. Keep the printed states beside the code as you work through the complete model below.

## Worked Example

### Put the Operations Behind a Store

`NoteStore` keeps its mutable list private. Callers use its public methods instead of reaching into the list directly. A `private val` list can still change its contents; `val` prevents replacing that binding, and `private` controls access.

1. `Note` records the ID and title as constructor properties.
2. `mutableListOf<Note>()` creates the initially empty typed list, using the form taught in Lesson 4.
3. `add` appends a supplied note. The caller is responsible for choosing a unique ID.
4. `find` returns `Note?`, because some IDs are absent. Its predicate compares IDs, not titles.
5. `remove` calls the store's `find`, checks the result, and removes a matching object from the list.
6. `update` uses the same check, then removes the old record and adds its copy. It preserves the ID and replaces only the named title.
7. The final calls add ID 1, update it, print its current title, remove it, and print the missing result.

`find(id)` calls our store method. `notes.find { ... }` calls the collection search. The receiver name makes their different roles clear.

In [ ]:
data class Note(val id: Int, val title: String)
class NoteStore {
    private val notes = mutableListOf<Note>()
    fun add(note: Note) { notes.add(note) }
    fun find(id: Int): Note? = notes.find { it.id == id }
    fun remove(id: Int) {
        val note = find(id)
        if (note != null) { notes.remove(note) }
    }
    fun update(id: Int, title: String) {
        val note = find(id)
        if (note != null) {
            notes.remove(note)
            notes.add(note.copy(title = title))
        }
    }
}
val store = NoteStore()
store.add(Note(1, "Lab ideas"))
store.update(1, "Android lab ideas")
println(store.find(1)?.title ?: "Missing")
store.remove(1)
println(store.find(1)?.title ?: "Missing")

The output is:

```text
Android lab ideas
Missing
```

The first line proves that a search after the update sees the replacement title. The second proves that a search after removal no longer finds ID 1. These two lines are useful checks, but do not prove every rule on their own. We also need tests for missing requests and another record that must stay unchanged.

| Step | Record with ID 1 | Meaning |
| --- | --- | --- |
| New store | Absent | The list begins empty |
| Add | `Lab ideas` | One record was appended |
| Update | `Android lab ideas` | A replacement keeps ID 1 |
| First find | Present | The current title is printed |
| Remove | Absent | The matching record leaves the list |
| Second find | Missing | Null becomes the display message |

### Check Behavior, Not Just One Happy Path

A useful test describes a starting state, an operation, and the result you expect. In your reminder model, test an empty store, an existing ID, a missing ID, and a full sequence of changes. Keep a second record so you can check that edits and removals target only the requested ID.

Use printed searches to compare the result with your prediction. Do not add code that merely prints the expected words without using the store. Create a fresh store at the start of each complete test sequence so an earlier run does not supply hidden state.

## Guided Practice

### Trace records with equal titles

Use the `Note` and `NoteStore` declarations from the worked example above. Before running the next cell, predict its three output lines. Explain which record changes, why the other record keeps its title, and what removing ID 99 does. Both titles begin as `Review`; the IDs are different.

In [ ]:
Your prediction:
Three output lines:
Which record changes and why:
What happens to the other record:
Effect of removing ID 99:


In [ ]:
val traceStore = NoteStore()
traceStore.add(Note(21, "Review"))
traceStore.add(Note(22, "Review"))
traceStore.update(21, "Review Kotlin")
traceStore.remove(99)
println(traceStore.find(21)?.title ?: "Missing")
println(traceStore.find(22)?.title ?: "Missing")
println(traceStore.find(99)?.title ?: "Missing")

<details>
<summary>Show answer</summary>

```text
Review Kotlin
Review
Missing
```

The update finds ID 21 and replaces that record with a copy whose title is `Review Kotlin`. ID 22 stays `Review`, even though its original title was identical. IDs identify records; equal titles do not make two records the same record. Removing absent ID 99 makes no change, and searching for it returns null, so Elvis displays `Missing`.

</details>

### Complete a safe lookup helper

Using the worked `NoteStore`, write `fun titleFor(store: NoteStore, id: Int): String`. Return the found note's title, or `Missing` when the ID is absent. Keep printing outside the helper.

Create a fresh `lookupStore`, add `Note(31, "Plan lab")`, and print helper results for IDs 31 and 90. Expect:

```text
Plan lab
Missing
```

Then change only the fallback to `No note` and rerun both checks. The first line stays `Plan lab`; the second becomes `No note`.

In [ ]:
// TODO: Define titleFor, create a fresh lookupStore, and test present/missing IDs.
// Then modify only the fallback text and rerun.


<details>
<summary>Show answer</summary>

```kotlin
fun titleFor(store: NoteStore, id: Int): String =
    store.find(id)?.title ?: "Missing"
val lookupStore = NoteStore()
lookupStore.add(Note(31, "Plan lab"))
println(titleFor(lookupStore, 31))
println(titleFor(lookupStore, 90))
```

`find` returns a nullable Note. The safe call reads the title only when a note was found. Elvis supplies display text for the missing case. For the modification, replace only `"Missing"` with `"No note"`. This helper reads the store; it does not add, update, or remove any records.

</details>

### Repair a discarded update

This intentionally faulty program runs, but it prints `Read` instead of the required updated title:

```kotlin
data class DebugReminder(val id: Int, val title: String)
val debugReminders = mutableListOf(DebugReminder(1, "Read"), DebugReminder(2, "Sketch"))
val debugMatch = debugReminders.find { it.id == 1 }
if (debugMatch != null) {
    debugMatch.copy(title = "Read Kotlin")
}
println(debugReminders.find { it.id == 1 }?.title ?: "Missing")
println(debugReminders.find { it.id == 2 }?.title ?: "Missing")
```

Explain what happens to the result of `copy`. Rewrite the snippet so it stores the replacement for ID 1 and preserves ID 2. Use the taught `find`, null check, `remove`, `add`, and `copy` operations. Expected corrected output:

```text
Read Kotlin
Sketch
```

In [ ]:
Your diagnosis:
What happens to the copy result:
Why the stored title stays Read:
How the repair preserves ID 2:


In [ ]:
// TODO: Rewrite the faulty snippet so the changed copy replaces only ID 1.
// Print both records to check the update and preservation.


<details>
<summary>Show answer</summary>

```kotlin
data class DebugReminder(val id: Int, val title: String)
val debugReminders = mutableListOf(DebugReminder(1, "Read"), DebugReminder(2, "Sketch"))
val debugMatch = debugReminders.find { it.id == 1 }
if (debugMatch != null) {
    debugReminders.remove(debugMatch)
    debugReminders.add(debugMatch.copy(title = "Read Kotlin"))
}
println(debugReminders.find { it.id == 1 }?.title ?: "Missing")
println(debugReminders.find { it.id == 2 }?.title ?: "Missing")
```

The faulty code creates a copy but discards its result. The original list still holds the unchanged record. The repair removes the found original and adds the returned copy. Because the copy changes only title, it keeps ID 1. Only that found record is removed, so ID 2 remains `Sketch`. Do not replace this with a change to every record or a change based on title equality. Removing and adding can change list order; these checks use IDs instead of depending on positions.

</details>

## Independent Practice

### Build and test a reminder store

Build an in-memory model using the Kotlin features from this module. Start from these behavior rules rather than copying the notes model unchanged:

- A `Reminder` data class has `id: Int`, `title: String`, and `completed: Boolean = false`, all `val` properties.
- `ReminderStore` owns a private mutable list. Its public methods are `add(reminder: Reminder)`, `find(id: Int): Reminder?`, `update(id: Int, title: String, completed: Boolean)`, and `remove(id: Int)`.
- The caller supplies unique IDs for new records. Duplicate-ID handling is outside this task.
- `update` changes both title and completion state for the matching ID, while keeping its ID. Other records remain unchanged.
- Missing updates and removals do nothing. Missing searches return null.
- Keep the model separate from printing. For checks, display a found reminder as `101: Read / false`, and a missing result as `Missing`. A small helper can avoid repeating this formatting.

Run the following sequence in one fresh store. Print the listed searches after each action:

| Action | Search IDs to print, in order |
| --- | --- |
| Create an empty store | 101 |
| Add ID 101, title Read, unfinished; add ID 202, title Pack, unfinished | 101 |
| Update ID 101 to Read Kotlin and completed true | 101, 202 |
| Update ID 999 to Ghost and true; remove ID 999 | 999, 101, 202 |
| Remove ID 101 | 101, 202 |
| Remove ID 202, then remove ID 202 again | 202 |

The ten printed lines must be:

```text
Missing
101: Read / false
101: Read Kotlin / true
202: Pack / false
Missing
101: Read Kotlin / true
202: Pack / false
Missing
202: Pack / false
Missing
```

Add two short boundary checks with fresh stores:

1. On an empty store, update and remove ID 404, then search for it. Print `Missing`; no record should be created and nothing should fail.
2. Add ID 301, title Practice, completed true, plus ID 302, title Walk, unfinished. Update ID 301 to Practice again and completed false. Print both searches; expect:

```text
301: Practice again / false
302: Walk / false
```

This second check confirms that update can reopen a completed reminder and preserves the other record. No Android UI, database, file save, or server connection is part of this model.

In [ ]:
// TODO: Build Reminder and ReminderStore from the behavior rules.
// Run the full sequence and the two boundary checks with fresh stores.


<details>
<summary>Show answer</summary>

```kotlin
data class Reminder(val id: Int, val title: String, val completed: Boolean = false)
class ReminderStore {
    private val reminders = mutableListOf<Reminder>()
    fun add(reminder: Reminder) {
        reminders.add(reminder)
    }
    fun find(id: Int): Reminder? = reminders.find { it.id == id }
    fun update(id: Int, title: String, completed: Boolean) {
        val reminder = find(id)
        if (reminder != null) {
            reminders.remove(reminder)
            reminders.add(reminder.copy(title = title, completed = completed))
        }
    }
    fun remove(id: Int) {
        val reminder = find(id)
        if (reminder != null) {
            reminders.remove(reminder)
        }
    }
}
fun reminderText(reminder: Reminder?): String =
    reminder?.let { "${it.id}: ${it.title} / ${it.completed}" } ?: "Missing"
val reminderStore = ReminderStore()
println(reminderText(reminderStore.find(101)))
reminderStore.add(Reminder(101, "Read"))
reminderStore.add(Reminder(202, "Pack"))
println(reminderText(reminderStore.find(101)))
reminderStore.update(101, "Read Kotlin", true)
println(reminderText(reminderStore.find(101)))
println(reminderText(reminderStore.find(202)))
reminderStore.update(999, "Ghost", true)
reminderStore.remove(999)
println(reminderText(reminderStore.find(999)))
println(reminderText(reminderStore.find(101)))
println(reminderText(reminderStore.find(202)))
reminderStore.remove(101)
println(reminderText(reminderStore.find(101)))
println(reminderText(reminderStore.find(202)))
reminderStore.remove(202)
reminderStore.remove(202)
println(reminderText(reminderStore.find(202)))
```

The private list holds records; the caller supplies their unique IDs. `find` searches by ID. Both changing methods guard the nullable result, so a missing ID cannot create or remove a different record. Update copies both requested fields while leaving the ID unchanged, then stores that replacement. Searching the other ID after each operation checks preservation.

For the empty-store boundary, create a new store, call `update(404, "Ghost", true)` and `remove(404)`, then format `find(404)`. It is still `Missing`.

For the reopen boundary, create another store, add `Reminder(301, "Practice", completed = true)` and `Reminder(302, "Walk")`, then call `update(301, "Practice again", false)`. The two searches print:

```text
301: Practice again / false
302: Walk / false
```

That case catches an update that always sets completed to true. The helper only formats results; printing expected strings without implementing the operations does not satisfy the behavior. List order may change when an update removes and appends a copy, so the tests use record IDs. The model stores values only while its in-memory store exists.

</details>

### Explain your evidence

After running your checks, identify the output that shows a successful update, a successful removal, and preservation of another record. Explain why the missing-ID checks must verify existing records too. Describe how the reopen test catches a mistake that the first update test could miss.

In [ ]:
Your explanation after testing:
Successful update evidence:
Successful removal evidence:
Other-record preservation evidence:
Why missing-ID tests also inspect existing records:
What the reopen test catches:


<details>
<summary>Show answer</summary>

`101: Read Kotlin / true` shows that both requested fields changed. `Missing` after removing ID 101 shows the record is gone. `202: Pack / false` still appears after updates, missing-ID operations, and removal of ID 101. Checking only that ID 999 is missing would not detect a faulty operation that also damaged an existing record. The reopen result `301: Practice again / false` catches an update that always forces the completion flag to true.

</details>

## Summary

- A small model combines data records, private state, public methods, and collection operations.
- Search by a stable ID when titles may change. In this exercise, callers supply unique IDs; the store does not enforce uniqueness.
- A nullable search and an explicit check handle missing records safely. Missing update and remove requests leave the store unchanged.
- Replacing a record with `copy` changes what the list contains without mutating the old data object. Removing and appending can change list order.
- Test normal operations, missing IDs, preservation of other records, and the full add-update-find-remove sequence.
- This model is in memory only. An Android screen and persistent storage are separate work.

You have completed the Kotlin crash course. In the next module, use Android Studio to begin connecting language-level behavior to an Android app.

## Reflection

A real reminders screen offers edit and delete actions. Explain why each action should carry the reminder ID even when two reminders have the same title. Identify one part of your model that could be reused behind that screen. Finally, explain what would still be needed to keep reminders after this in-memory model is discarded; do not write new APIs.

In [ ]:
Your reflection:
Why actions should use record IDs:
Reusable model behavior:
What is still needed beyond in-memory storage:


<details>
<summary>Show answer</summary>

Titles can repeat or change, so an edit or delete should target the intended record by ID. The store's add, find, update, and remove rules can be reused behind a screen without depending on how the screen is drawn. Keeping reminders after the model is discarded requires a separate storage mechanism, such as device storage or a backend, and a way to load the data again. This notebook implements neither persistence nor an Android interface.

</details>

## Supplemental Reading

- [Kotlin basic syntax](https://kotlinlang.org/docs/basic-syntax.html) — review function, class, property, and control-flow syntax used in the model.
- [Kotlin null safety](https://kotlinlang.org/docs/null-safety.html) — review nullable results, explicit checks, safe calls, and fallback expressions.
- [Kotlin data classes](https://kotlinlang.org/docs/data-classes.html) — review generated value operations and copies with changed properties.
- [Kotlin collections overview](https://kotlinlang.org/docs/collections-overview.html) — review list types and mutable collection operations.